# ProAg Track 2 — LLM Guardrails

Implements §5 of the project brief end-to-end:

1. **Anonymization** — strip producer/vendor names → tokens; restore on return
2. **Computed-metrics-only** — never send raw rows; only 5–10 numbers per cycle
3. **Number verification** — LLM cites every number; code rejects mismatches
4. **Hard refusals** — price predictions, hedging recs, cross-producer comparisons
5. **Audit logging** — every prompt + response logged with user + producer ID
6. **Function whitelist** — LLM can only request a defined set of metric functions

**Repo layout this notebook expects:**
```
/workspaces/CCDS/
├── Data/                      ← raw producer CSVs
├── Outputs/                   ← pipeline outputs (the CSVs from proag_pipeline.ipynb)
│   ├── fact_cycle_pnl.csv
│   ├── dim_cycle.csv
│   ├── fact_hedge_pnl.csv
│   ├── fact_anomalies.csv
│   └── llm_audit_log.jsonl    ← created by this notebook
├── proag_pipeline.ipynb
└── llm_guardrails.ipynb       ← this file
```

**Run order:** Run `proag_pipeline.ipynb` FIRST so the CSVs in `Outputs/` exist. Then run this notebook.

In [2]:
pip install pandas anthropic


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Setup

In [3]:
import json
import os
import re
import time
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

# ---- Paths in the codespace ----
DATA_DIR   = Path("/workspaces/CCDS/Data")
OUTPUT_DIR = Path("/workspaces/CCDS/Outputs")
AUDIT_LOG  = OUTPUT_DIR / "llm_audit_log.jsonl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Anthropic API key — only needed for live calls. Without it, every cell still
# works because we use a deterministic fallback that returns a stub summary.
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")  # or paste a string here
ANTHROPIC_MODEL   = "claude-sonnet-4-20250514"

print(f"Data dir:   {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Audit log:  {AUDIT_LOG}")
print(f"API key:    {'set' if ANTHROPIC_API_KEY else 'not set (will use fallback)'}")

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"Output folder not found at {OUTPUT_DIR}.\n"
        "Run proag_pipeline.ipynb first so it writes the CSVs there."
    )
missing = [n for n in [
    "fact_cycle_pnl.csv", "dim_cycle.csv",
    "fact_hedge_pnl.csv", "fact_anomalies.csv",
] if not (OUTPUT_DIR / n).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing pipeline outputs in {OUTPUT_DIR}: {missing}.\n"
        "Re-run proag_pipeline.ipynb so it writes these."
    )

Data dir:   /workspaces/CCDS/Data
Output dir: /workspaces/CCDS/Outputs
Audit log:  /workspaces/CCDS/Outputs/llm_audit_log.jsonl
API key:    not set (will use fallback)


## 2. Anonymization (Guardrail #2 from §5)

Producer names, vendor names, and account numbers become tokens like `<PRODUCER_A>` before any prompt leaves our environment. The LLM provider's servers never see real identities.

In [4]:
@dataclass
class TokenMap:
    forward: dict = field(default_factory=dict)   # real -> token
    reverse: dict = field(default_factory=dict)   # token -> real

    def add(self, real: str, kind: str) -> str:
        if real in self.forward:
            return self.forward[real]
        prefix = kind.upper()
        n = sum(1 for v in self.forward.values() if v.startswith(f"<{prefix}_"))
        token = f"<{prefix}_{chr(ord('A') + n)}>"
        self.forward[real] = token
        self.reverse[token] = real
        return token

    def anonymize(self, text: str) -> str:
        out = text
        # Replace longest first to avoid prefix collisions
        for real in sorted(self.forward, key=len, reverse=True):
            out = re.sub(re.escape(real), self.forward[real], out)
        return out

    def restore(self, text: str) -> str:
        out = text
        for token, real in self.reverse.items():
            out = out.replace(token, real)
        return out


def build_token_map(producers=None, vendors=None) -> TokenMap:
    tm = TokenMap()
    for p in producers or []:
        if p:
            tm.add(p, "PRODUCER")
    for v in vendors or []:
        if v:
            tm.add(v, "VENDOR")
    return tm


# --- self-test ---
tm = build_token_map(
    producers=["Demo Producer A", "Demo Producer B"],
    vendors=["VetHealth Inc"],
)
raw = "Demo Producer A paid VetHealth Inc. Demo Producer B did not."
anon = tm.anonymize(raw)
restored = tm.restore(anon)

print("Original:", raw)
print("Anonymized:", anon)
print("Restored: ", restored)
assert "Demo Producer A" not in anon and "VetHealth Inc" not in anon
assert restored == raw
print("\n✓ anonymization round-trip works")

Original: Demo Producer A paid VetHealth Inc. Demo Producer B did not.
Anonymized: <PRODUCER_A> paid <VENDOR_A>. <PRODUCER_B> did not.
Restored:  Demo Producer A paid VetHealth Inc. Demo Producer B did not.

✓ anonymization round-trip works


## 3. Function whitelist (Guardrail: "No direct database access")

From §5: *The LLM only calls defined functions like `get_cycle_pnl(cycle_id)`. The producer ID is set on the server side; the LLM cannot ask for someone else's data.*

Below: every function the LLM is allowed to call. Each returns a small dict of pre-computed numbers — never raw rows.

In [5]:
# Load the pipeline outputs once
_pnl = pd.read_csv(OUTPUT_DIR / "fact_cycle_pnl.csv")
_cycles = pd.read_csv(OUTPUT_DIR / "dim_cycle.csv")
_hedge = pd.read_csv(OUTPUT_DIR / "fact_hedge_pnl.csv")
_anomalies = pd.read_csv(OUTPUT_DIR / "fact_anomalies.csv")


def get_cycle_pnl(cycle_id: str) -> dict:
    """Return computed P&L metrics for a single cycle. Numbers only."""
    row = _pnl[_pnl["cycle_id"] == cycle_id]
    if row.empty:
        raise ValueError(f"Unknown cycle_id: {cycle_id}")
    r = row.iloc[0]
    return {
        "cycle_id": cycle_id,
        "received_head": int(r["received_head"]),
        "packer_revenue": round(float(r["packer_revenue"]), 2),
        "total_cost": round(float(r["total_cost"]), 2),
        "hedge_pnl": round(float(r["hedge_pnl"]), 2),
        "net_pnl": round(float(r["net_pnl"]), 2),
        "pnl_per_head": round(float(r["pnl_per_head"]), 2)
            if pd.notna(r["pnl_per_head"]) else None,
    }


def get_cycle_anomalies(cycle_id: str) -> dict:
    rows = _anomalies[_anomalies["cycle_id"] == cycle_id]
    return {
        "cycle_id": cycle_id,
        "flag_count": int(len(rows)),
        "severities": rows["severity"].value_counts().to_dict() if not rows.empty else {},
    }


def get_hedge_summary(cycle_id: str) -> dict:
    rows = _hedge[_hedge["cycle_id"] == cycle_id]
    if rows.empty:
        return {"cycle_id": cycle_id, "contracts": 0, "total_hedge_pnl": 0.0}
    return {
        "cycle_id": cycle_id,
        "contracts": int(len(rows)),
        "total_head_covered": int(rows["head_covered"].sum()),
        "avg_strike_cwt": round(float(rows["strike_cwt"].mean()), 2),
        "avg_settlement_cwt":
            round(float(rows["settlement_cwt"].mean()), 2)
            if rows["settlement_cwt"].notna().any() else None,
        "total_hedge_pnl": round(float(rows["hedge_gain_loss"].sum()), 2),
    }


# Whitelist — the LLM may only ask for these
ALLOWED_FUNCTIONS = {
    "get_cycle_pnl": get_cycle_pnl,
    "get_cycle_anomalies": get_cycle_anomalies,
    "get_hedge_summary": get_hedge_summary,
}

# Demo
demo_cycle = _pnl["cycle_id"].iloc[0]
print(f"get_cycle_pnl({demo_cycle!r}):")
print(json.dumps(get_cycle_pnl(demo_cycle), indent=2))

get_cycle_pnl('PG-1017'):
{
  "cycle_id": "PG-1017",
  "received_head": 2285,
  "packer_revenue": 0.0,
  "total_cost": 0.0,
  "hedge_pnl": -38107.56,
  "net_pnl": -38107.56,
  "pnl_per_head": -16.68
}


## 4. Hard refusals (Guardrail: prohibited request types)

From §5: *Hard refusals for: price predictions, hedging recommendations, cross-producer comparisons (unless explicit opt-in).*

Pattern matching is the first line of defense — caught before the LLM even sees the request.

In [6]:
PROHIBITED_PATTERNS = [
    # Price predictions
    (r"\b(predict|forecast|will|should)\b.*\b(price|hog|corn|cwt|market)\b",
     "Price prediction"),
    (r"\b(price|hog|cwt)\b.*\b(go up|go down|rise|fall|next week|next month)\b",
     "Price prediction"),
    # Hedging recommendations
    (r"\b(should|recommend|suggest|advise)\b.*\b(hedge|sell|buy|cover|lock)\b",
     "Hedging recommendation"),
    (r"\b(when|how much|what)\b.*\b(should|to)\b.*\b(hedge|cover)\b",
     "Hedging recommendation"),
    # Cross-producer comparison
    (r"\bcompare\b.*\b(producer|operation|farm)\b", "Cross-producer comparison"),
    (r"\bother\s+producer", "Cross-producer comparison"),
]


def check_prohibited(user_message: str) -> str | None:
    text = user_message.lower()
    for pattern, label in PROHIBITED_PATTERNS:
        if re.search(pattern, text):
            return label
    return None


# --- self-test ---
test_cases = [
    ("Will hog prices go up next month?",          "Price prediction"),
    ("Should we hedge more for cycle PG-1014?",    "Hedging recommendation"),
    ("Compare PG-1017 with other producers.",       "Cross-producer comparison"),
    ("Summarize cycle PG-1014.",                    None),
    ("What was the net P&L for PG-1016?",          None),
]
for msg, expected in test_cases:
    got = check_prohibited(msg)
    mark = "✓" if got == expected else "✗"
    print(f"  {mark} {msg!r:<55} -> {got}")

  ✓ 'Will hog prices go up next month?'                     -> Price prediction
  ✓ 'Should we hedge more for cycle PG-1014?'               -> Hedging recommendation
  ✓ 'Compare PG-1017 with other producers.'                 -> Cross-producer comparison
  ✓ 'Summarize cycle PG-1014.'                              -> None
  ✓ 'What was the net P&L for PG-1016?'                     -> None


## 5. Audit log (Guardrail: "Logging")

From §5: *Every prompt, retrieved data, and response is logged with the user and producer ID — forensic trail for any incident.*

Append-only JSONL — one line per event. Easy to grep, easy to replay.

In [7]:
def audit_log(event: dict) -> None:
    """Append a single event to the audit log as JSONL."""
    AUDIT_LOG.parent.mkdir(parents=True, exist_ok=True)
    event_with_ts = {"timestamp": time.time(), **event}
    with AUDIT_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(event_with_ts, default=str) + "\n")


def read_audit_log() -> pd.DataFrame:
    if not AUDIT_LOG.exists():
        return pd.DataFrame()
    rows = []
    with AUDIT_LOG.open() as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


# Demo
audit_log({
    "event": "setup", "user": "advisor_jane",
    "producer": demo_cycle, "action": "guardrails_initialized",
})
print("Audit log entries:")
print(read_audit_log().to_string(index=False))

Audit log entries:
   timestamp event         user producer                 action
1.777544e+09 setup advisor_jane  PG-1017 guardrails_initialized


## 6. Number verification (Guardrail: "Every number is cited and verified")

From §5: *The LLM tags each number with its source. Our code checks that the cited value matches the deterministic computation. Mismatches block the response.*

We require the LLM to wrap every number in `[[key=value]]` tags using only keys from the metrics dict we sent. The verifier strips them after checking values match.

In [8]:
CITATION_RE = re.compile(r"\[\[(\w+)=([\w\-\d\.,]+)\]\]")


def verify_citations(text: str, metrics: dict, tolerance: float = 0.01):
    """Verify every [[key=value]] tag matches the metrics dict.

    Returns:
      (clean_text, ok, errors)
        clean_text — text with tags replaced by their value
        ok         — True if every tag matched and at least one tag was found
        errors     — list of mismatch descriptions
    """
    errors = []
    found = 0

    def _replace(match):
        nonlocal found
        found += 1
        key = match.group(1)
        cited = match.group(2)
        if key not in metrics:
            errors.append(f"Cited unknown key: {key}")
            return match.group(0)
        actual = metrics[key]
        if actual is None:
            errors.append(f"Cited {key} but actual value is None")
            return match.group(0)
        # String values (cycle_id etc.) compared as strings
        if isinstance(actual, str):
            if cited != actual:
                errors.append(f"Mismatch for {key}: cited {cited!r}, actual {actual!r}")
                return match.group(0)
            return cited
        # Numeric values — parse and compare with tolerance
        try:
            cited_num = float(cited.replace(",", ""))
        except ValueError:
            errors.append(f"Could not parse number for {key}: {cited!r}")
            return match.group(0)
        if abs(cited_num - float(actual)) > tolerance * max(1.0, abs(float(actual))):
            errors.append(
                f"Mismatch for {key}: cited {cited_num}, actual {actual}"
            )
            return match.group(0)
        # Clean substitution — drop the tag, keep the value
        return cited

    clean = CITATION_RE.sub(_replace, text)
    ok = found > 0 and not errors
    return clean, ok, errors


# --- self-test ---
metrics = {"cycle_id": "PG-1016", "received_head": 2399, "net_pnl": 52234.25}

good = (
    "Cycle [[cycle_id=PG-1016]] received [[received_head=2399]] head and "
    "finished with $[[net_pnl=52234.25]] net P&L."
)
clean, ok, errs = verify_citations(good, metrics)
print("GOOD:", clean, "   ok=", ok, "   errs=", errs)

bad = "Cycle PG-1016 finished with $[[net_pnl=99999.00]] — clearly wrong."
clean, ok, errs = verify_citations(bad, metrics)
print("BAD: ok=", ok, "   errs=", errs)

uncited = "Cycle PG-1016 finished with $52,234.25 net P&L (no tags)."
clean, ok, errs = verify_citations(uncited, metrics)
print("UNCITED: ok=", ok, "   errs=", errs, "   (no tags = blocked)")

GOOD: Cycle PG-1016 received 2399 head and finished with $52234.25 net P&L.    ok= True    errs= []
BAD: ok= False    errs= ['Mismatch for net_pnl: cited 99999.0, actual 52234.25']
UNCITED: ok= False    errs= []    (no tags = blocked)


## 7. Prompt template + LLM client

Single whitelisted prompt. Forces the LLM to use citation tags so the verifier can check them. Anonymizes outgoing text and restores names on the response.

In [9]:
CYCLE_SUMMARY_PROMPT = """\
You are an analyst writing a one-paragraph briefing for a producer-advisor call.

Rules — non-negotiable:
1. Use ONLY the numbers in the metrics block below. Never invent or estimate.
2. Wrap every number you mention in citation tags: [[key=value]]
   Example: "received [[received_head=2399]] head"
   Use the exact key from the metrics block and the exact numeric value.
3. Do NOT predict prices, recommend trades, or speculate beyond the data.
4. Keep it to 3-4 sentences.
5. If a metric is null/missing, omit it. Do not guess.

Metrics:
{metrics}

Write the summary now.
"""


def _fallback_summary(metrics: dict) -> str:
    """Deterministic stub used when no API key is set. Uses citation tags so it
    passes verification just like a real LLM response would."""
    parts = [f"Cycle [[cycle_id={metrics['cycle_id']}]]"]
    if metrics.get("received_head") is not None:
        parts.append(f"received [[received_head={metrics['received_head']}]] head")
    if metrics.get("net_pnl") is not None:
        parts.append(f"and finished with $[[net_pnl={metrics['net_pnl']}]] net P&L")
    if metrics.get("pnl_per_head") is not None:
        parts.append(f"([[pnl_per_head={metrics['pnl_per_head']}]] per head)")
    return " ".join(parts) + "."


def call_llm(prompt: str) -> str:
    """Call Claude. If no key is set, return the fallback so the rest of the
    notebook still runs end-to-end."""
    if not ANTHROPIC_API_KEY:
        return None  # signal: caller should use fallback
    try:
        from anthropic import Anthropic
    except ImportError:
        print("anthropic not installed — pip install anthropic to enable live LLM")
        return None
    client = Anthropic(api_key=ANTHROPIC_API_KEY)
    msg = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(
        b.text for b in msg.content if getattr(b, "type", "") == "text"
    ).strip()


print("Prompt template + LLM client ready.")

Prompt template + LLM client ready.


## 8. Full guardrailed pipeline

This is the public function. Everything from §5 runs here in the right order:

1. Check prohibited request types → refuse if matched
2. Look up the producer → only the metrics they're allowed to see
3. Anonymize names
4. Send to LLM
5. Restore names
6. Verify every cited number
7. Log everything

In [10]:
def summarize_cycle_safely(
    user: str,
    cycle_id: str,
    user_request: str = "Summarize this cycle for the advisor.",
    producer_name: str | None = None,
    vendor_names: list | None = None,
) -> dict:
    """Generate a guardrailed cycle summary.

    Returns a dict with: status, summary (or refusal_reason), metrics_sent,
    verification_errors. Every step is audit-logged.
    """
    base_event = {"user": user, "producer": cycle_id, "request": user_request}

    # 1. Prohibited request check
    refusal = check_prohibited(user_request)
    if refusal:
        audit_log({**base_event, "event": "refused", "reason": refusal})
        return {
            "status": "refused",
            "refusal_reason": refusal,
            "summary": None,
        }

    # 2. Look up metrics via the whitelist (no direct DB access)
    try:
        metrics = get_cycle_pnl(cycle_id)
    except ValueError as e:
        audit_log({**base_event, "event": "error", "detail": str(e)})
        return {"status": "error", "summary": None, "refusal_reason": str(e)}

    # 3. Anonymize
    tm = build_token_map(
        producers=[producer_name] if producer_name else [],
        vendors=vendor_names or [],
    )
    safe_metrics = json.loads(tm.anonymize(json.dumps(metrics)))
    prompt = CYCLE_SUMMARY_PROMPT.format(metrics=json.dumps(safe_metrics, indent=2))

    audit_log({
        **base_event,
        "event": "prompt_sent",
        "metrics_keys": list(metrics.keys()),
        "anonymized": list(tm.forward.values()),
    })

    # 4. Call LLM (or fallback)
    raw_response = call_llm(prompt)
    if raw_response is None:
        raw_response = _fallback_summary(safe_metrics)
        used_fallback = True
    else:
        used_fallback = False

    # 5. Restore names
    response = tm.restore(raw_response)

    # 6. Verify citations against the ORIGINAL metrics
    clean, ok, errors = verify_citations(response, metrics)

    # 7. Final audit
    audit_log({
        **base_event,
        "event": "response",
        "used_fallback": used_fallback,
        "verification_ok": ok,
        "verification_errors": errors,
    })

    if not ok:
        return {
            "status": "blocked",
            "refusal_reason": "Citation verification failed",
            "verification_errors": errors,
            "raw_response": response,
            "metrics_sent": metrics,
        }

    return {
        "status": "ok",
        "summary": clean,
        "metrics_sent": metrics,
        "used_fallback": used_fallback,
    }


print("Public function summarize_cycle_safely ready.")

Public function summarize_cycle_safely ready.


## 9. Demo runs — happy path + every refusal type

In [11]:
print("=" * 70)
print(" HAPPY PATH — legitimate summary request")
print("=" * 70)
result = summarize_cycle_safely(
    user="advisor_jane",
    cycle_id=demo_cycle,
    user_request="Give me a one-paragraph summary of this cycle for the call.",
    producer_name="Demo Producer A",
    vendor_names=["VetHealth Inc", "Iowa Feed Mill"],
)
print(json.dumps(result, indent=2, default=str))

 HAPPY PATH — legitimate summary request
{
  "status": "ok",
  "summary": "Cycle PG-1017 received 2285 head and finished with $-38107.56 net P&L (-16.68 per head).",
  "metrics_sent": {
    "cycle_id": "PG-1017",
    "received_head": 2285,
    "packer_revenue": 0.0,
    "total_cost": 0.0,
    "hedge_pnl": -38107.56,
    "net_pnl": -38107.56,
    "pnl_per_head": -16.68
  },
  "used_fallback": true
}


In [12]:
print("=" * 70)
print(" REFUSAL #1 — price prediction")
print("=" * 70)
print(json.dumps(
    summarize_cycle_safely(
        user="advisor_jane", cycle_id=demo_cycle,
        user_request="Will hog prices go up next month?",
    ),
    indent=2,
))

print()
print("=" * 70)
print(" REFUSAL #2 — hedging recommendation")
print("=" * 70)
print(json.dumps(
    summarize_cycle_safely(
        user="advisor_jane", cycle_id=demo_cycle,
        user_request="Should we hedge more aggressively for this cycle?",
    ),
    indent=2,
))

print()
print("=" * 70)
print(" REFUSAL #3 — cross-producer comparison")
print("=" * 70)
print(json.dumps(
    summarize_cycle_safely(
        user="advisor_jane", cycle_id=demo_cycle,
        user_request="Compare this cycle with other producers.",
    ),
    indent=2,
))

 REFUSAL #1 — price prediction
{
  "status": "refused",
  "refusal_reason": "Price prediction",
  "summary": null
}

 REFUSAL #2 — hedging recommendation
{
  "status": "refused",
  "refusal_reason": "Hedging recommendation",
  "summary": null
}

 REFUSAL #3 — cross-producer comparison
{
  "status": "refused",
  "refusal_reason": "Cross-producer comparison",
  "summary": null
}


## 10. Audit log review

Forensic trail. Every prompt, refusal, and verification result is here.

In [13]:
log_df = read_audit_log()
print(f"Total events: {len(log_df)}")
print()
log_df.tail(10)

Total events: 6



,timestamp,event,user,producer,action,request,metrics_keys,anonymized,used_fallback,verification_ok,verification_errors,reason
0,1.777544e+09,setup,advisor_jane,PG-1017,guardrails_initialized,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.777544e+09,prompt_sent,advisor_jane,PG-1017,NaN,Give me a one-paragraph summary of this cycle ...,"[cycle_id, received_head, packer_revenue, tota...","[<PRODUCER_A>, <VENDOR_A>, <VENDOR_B>]",NaN,NaN,NaN,NaN
2,1.777544e+09,response,advisor_jane,PG-1017,NaN,Give me a one-paragraph summary of this cycle ...,NaN,NaN,True,True,[],NaN
3,1.777544e+09,refused,advisor_jane,PG-1017,NaN,Will hog prices go up next month?,NaN,NaN,NaN,NaN,NaN,Price prediction
4,1.777544e+09,refused,advisor_jane,PG-1017,NaN,Should we hedge more aggressively for this cycle?,NaN,NaN,NaN,NaN,NaN,Hedging recommendation
5,1.777544e+09,refused,advisor_jane,PG-1017,NaN,Compare this cycle with other producers.,NaN,NaN,NaN,NaN,NaN,Cross-producer comparison
